# Cargar OHLCV de todas las coins desde Qlib

Este notebook carga la serie de precios (Open/High/Low/Close/Volume) de **todas las coins** del universo crypto directamente desde el dataset **Qlib** (no desde CSV), usando `qlib.data.D.features()`.

- **Datos:** `data/qlib` (formato binario Qlib `.bin`)
- **Universo:** `data/qlib/instruments/crypto.txt`
- **Campo por defecto muestra:** OHLCV (`$open`, `$high`, `$low`, `$close`, `$volume`)

> Requiere el kernel Python del venv `qlib-venv` (donde está instalado `qlib`).

## 1. Inicializar Qlib

Inicializamos Qlib contra el dataset `data/qlib` (mismo provider y región que usa `qlib_sfm_pipeline.v8.py`).

In [1]:
from pathlib import Path

import qlib
from qlib.config import REG_US
from qlib.data import D

# --- Resuelve el repo raíz (/opt/data/qlib) aunque el notebook se abra
# desde el kernel en otra carpeta (p.ej. nbconvert, Jupyter Lab en otro dir)
PROJECT_ROOT = Path.cwd()
# Sube directorios si no estamos en la raíz del repo (presencia de data/qlib)
for _ in range(5):
    if (PROJECT_ROOT / "data" / "qlib" / "instruments").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROVIDER_URI = PROJECT_ROOT / "data" / "qlib"   # mismo provider/región que el pipeline SFM v8
FIELDS = ["$open", "$high", "$low", "$close", "$volume"]

qlib.init(provider_uri=str(PROVIDER_URI), region=REG_US, kernels=1)
print("Qlib inicializado OK →", PROVIDER_URI)

[109306:MainThread](2026-09-01 20:57:15,288) INFO - qlib.Initialization - [config.py:473] - default_conf: client.


[109306:MainThread](2026-09-01 20:57:17,134) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[109306:MainThread](2026-09-01 20:57:17,135) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/opt/data/qlib/data/qlib')}


Qlib inicializado OK → /opt/data/qlib/data/qlib


## 2. Universo de coins

Leemos el universo declarado desde `instruments/crypto.txt` (cada línea: `simbolo\tfecha_inicio\tfecha_fin`).

In [2]:
instruments_path = PROJECT_ROOT / "data" / "qlib" / "instruments" / "crypto.txt"
coins = []
for line in instruments_path.read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        coins.append(line.split("\t")[0])

print(f"{len(coins)} coins en el universo:")
print("  " + ", ".join(sorted(coins)))

9 coins en el universo:
  ada, btc, doge, eth, link, ltc, sol, xlm, xrp


## 3. Cargar cada coin en pandas y mostrar head/tail

Para **cada coin** cargamos su tabla OHLCV desde Qlib con `D.features()` y mostramos:
- **head(3)** → primeras 3 filas **reales** (desde la fecha de lanzamiento de la coin, no desde ceros de padding)
- **tail(3)** → últimas 3 filas
- rango de fechas, nº de filas y precio de apertura/cierre

> Nota: el dataset Qlib rellena con `0.0` los días previos al lanzamiento de cada coin (antes de su primera fecha en `crypto.txt`). Para que el `head` sea útil, cargamos cada coin desde su propia fecha de inicio.

In [3]:
# Fecha real de inicio de cada coin (desde crypto.txt): simbolo, inicio, fin
start_by_coin = {}
for line in instruments_path.read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        parts = line.split("\t")
        start_by_coin[parts[0]] = parts[1]

dataframes = {}

for coin in sorted(coins):
    start = start_by_coin[coin]
    df = D.features([coin], FIELDS, start_time=start, end_time="2026-12-31")
    # Suelta el nivel extra del MultiIndex por si solo hay un instrumento
    df = df.droplevel("instrument", axis=0) if "instrument" in df.index.names else df
    dataframes[coin] = df

    print("=" * 68)
    print(f"  {coin.upper()}")
    print("=" * 68)
    print(f"  Filas: {len(df)}  |  Rango: {df.index[0].date()} → {df.index[-1].date()}")
    display(df.head(3))
    print("  " + "·" * 60 + "  …")
    display(df.tail(3))
    print()

print(f"✅ Cargadas {len(dataframes)} coins en el dict `dataframes`.")

  ADA
  Filas: 3059  |  Rango: 2018-04-17 → 2026-08-31


,$open,$high,$low,$close,$volume
datetime,,,,,
2018-04-17,0.25551,0.2880,0.23983,0.24260,67462296.0
2018-04-18,0.24260,0.2646,0.24201,0.26200,31328096.0
2018-04-19,0.26199,0.2750,0.25777,0.27004,50859980.0


  ····························································  …


,$open,$high,$low,$close,$volume
datetime,,,,,
2026-08-29,0.203100,0.203500,0.198900,0.201800,46623256.0
2026-08-30,0.201800,0.206100,0.189400,0.192700,111173584.0
2026-08-31,0.212376,0.217739,0.190268,0.192917,0.0



  BTC
  Filas: 3302  |  Rango: 2017-08-17 → 2026-08-31


...


  XRP
  Filas: 3042  |  Rango: 2018-05-04 → 2026-08-31


,$open,$high,$low,$close,$volume
datetime,,,,,
2018-05-04,0.5000,1.500,0.50000,0.88990,20890214.0
2018-05-05,0.8898,0.935,0.88800,0.90280,16816166.0
2018-05-06,0.9028,0.918,0.83774,0.86483,16002036.0


  ····························································  …


,$open,$high,$low,$close,$volume
datetime,,,,,
2026-08-29,1.3838,1.4032,1.3768,1.3964,50379184.0
2026-08-30,1.3964,1.4335,1.3352,1.3575,94998016.0
2026-08-31,1.4200,1.4700,1.3400,1.3600,0.0



✅ Cargadas 9 coins en el dict `dataframes`.


## 4. (Opcional) Doble comprobación: head/tail de las 9 en una sola tabla

Si prefieres ver todas las coins de un vistazo, las cargamos desde su fecha de inicio real, concatenamos en un solo DataFrame (índice `instrument`/`datetime`) y mostramos `head(2)` y `tail(2)` combinados.

In [4]:
import pandas as pd

# Cargamos cada coin desde su fecha real de inicio (así no hay ceros de padding) y concatenamos
frames = [dataframes[c].copy() for c in coins]
for c, df in zip(coins, frames):
    df.index = pd.MultiIndex.from_product([[c.upper()], df.index], names=["instrument", "datetime"])
all_feats = pd.concat(frames)

print("SHAPE TOTAL:", all_feats.shape)
print("\n── HEAD (primeros 2 días de cada coin) ──")
print(all_feats.groupby(level="instrument").head(2))
print("\n── TAIL (últimos 2 días de cada coin) ──")
print(all_feats.groupby(level="instrument").tail(2))

SHAPE TOTAL: (26516, 5)

── HEAD (primeros 2 días de cada coin) ──
                             $open        $high         $low       $close  \
instrument datetime                                                         
ADA        2018-04-17     0.255510     0.288000     0.239830     0.242600   
           2018-04-18     0.242600     0.264600     0.242010     0.262000   
BTC        2017-08-17  4261.479980  4485.390137  4200.740234  4285.080078   
           2017-08-18  4285.080078  4371.520020  3938.770020  4108.370117   
DOGE       2019-07-05     0.004490     0.004600     0.003550     0.003870   
           2019-07-06     0.003874     0.003943     0.003365     0.003500   
ETH        2017-08-17   301.130005   312.179993   298.000000   302.000000   
           2017-08-18   302.000000   311.790009   283.940002   293.959991   
LINK       2019-01-16     0.535500     0.535500     0.466800     0.489500   
           2019-01-17     0.489500     0.495300     0.463900     0.475600   
LTC      